### 1. Configuração do Ecossistema de Inteligência Artificial e Deep Learning
Nesta célula inicial, estabelecemos a base tecnológica do projeto **PlaySafe4All**, integrando bibliotecas de processamento de dados e motores de aprendizagem profunda:

* **Manipulação de Dados (`Pandas` & `NumPy`)**: Essenciais para a estruturação das tabelas biomecânicas e operações matriciais.
* **Deep Learning com `TensorFlow` & `Keras`**: Introduzimos redes neuronais artificiais para captar padrões não-lineares complexos entre o tempo de exposição e o risco de lesão.
* **Gestão de Sistema (`OS` & `Time`)**: Permitem a organização de ficheiros e a monitorização da performance computacional dos modelos.
* **Ecossistema `Scikit-Learn`**: 
    * **Pré-processamento**: Ferramentas de divisão de dados e normalização (`StandardScaler`), vitais para a estabilidade das redes neuronais.
    * **Métricas de Validação**: Conjunto de indicadores para medir a eficácia do sistema, com foco prioritário no **Recall**.

In [14]:
# ==============================================================================
# CÉLULA 1: IMPORTAÇÃO DE BIBLIOTECAS
# Responsabilidade: Carregar todas as ferramentas necessárias para o projeto.
# ==============================================================================
import pandas as pd
import numpy as np
import os
import time
import tensorflow as tf


# Modelos e Métricas
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

print("✅ Todas as bibliotecas foram carregadas com sucesso!")

✅ Todas as bibliotecas foram carregadas com sucesso!


### 2. Aquisição e Consolidação de Dados Biomecânicos
Nesta fase, o sistema estabelece a ligação com a diretoria `Data` para importar os conjuntos de dados que servirão de base para o treino da Rede Neuronal.

* **Caminhos Robustos**: Utilizamos a biblioteca `os` para construir caminhos de ficheiros relativos. Isto assegura que o projeto **PlaySafe4All** é portátil, permitindo que qualquer examinador execute o código sem necessidade de configurar diretórios absolutos.
* **Segurança no Carregamento (`Try-Except`)**: Implementámos uma estrutura de tratamento de erros para capturar falhas de leitura. Caso os ficheiros CSV dos artigos científicos não estejam acessíveis, o sistema emite um alerta claro em vez de interromper a execução.
* **Verificação de Escala**: O sistema apresenta o total de amostras carregadas, fornecendo uma métrica inicial sobre o volume de dados que será utilizado para alimentar as camadas de entrada da rede.

In [15]:
# ==============================================================================
# CÉLULA 2: CARREGAMENTO DE DADOS
# Responsabilidade: Aceder à pasta 'Data' e carregar os datasets localmente.
# ==============================================================================
print("--- FASE 1: CARREGAMENTO ---")

# Caminho relativo (funciona em qualquer PC com a pasta Data ao lado do Notebook)
BASE_PATH = "../Data" 

try:
    # Construindo os caminhos de forma robusta
    df1 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 1.csv"), sep=',')
    df2 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 2.csv"), sep=',')
    df3 = pd.read_csv(os.path.join(BASE_PATH, "Base Artigo 3.csv"), sep=',')
    
    print("✅ Arquivos carregados com sucesso!")
    print(f"Total de amostras: {len(df1) + len(df2) + len(df3)}")
    load_successful = True
    
except FileNotFoundError as e:
    print(f"❌ ERRO: Não encontrei a pasta ou os ficheiros em: {BASE_PATH}")
    load_successful = False

--- FASE 1: CARREGAMENTO ---
✅ Arquivos carregados com sucesso!
Total de amostras: 181


### 3. Engenharia de Dados e Normalização para Deep Learning
Nesta fase, preparamos os dados biomecânicos para alimentar a Rede Neuronal. Ao contrário de modelos de árvore, as redes neuronais requerem que os dados estejam matematicamente equilibrados para garantir a convergência do treino.

* **Consolidação e Limpeza**: Unificamos os datasets e removemos variáveis não preditivas. O foco recai sobre métricas de exposição e performance (ex: `T0SRTMax`, `T0Veli`).
* **Tratamento de Dados Omissos**: Utilizamos a **Mediana** para preencher valores em falta, assegurando que o modelo não seja desviado por valores extremos (*outliers*).
* **Binarização de Risco**: O alvo (*Target*) é convertido para um formato 0/1, permitindo o uso da função de perda `binary_crossentropy` na rede.
* **Divisão Estratificada**: A partição de Treino/Teste mantém a proporção original de lesões, essencial para que a rede aprenda a reconhecer o evento raro (a entorse).
* **Padronização Crítica (`StandardScaler`)**: Transformamos todas as variáveis para uma escala comum (média 0, desvio padrão 1). Este passo é vital para as Redes Neuronais: sem normalização, os gradientes podem tornar-se instáveis, impedindo a rede de aprender padrões de força ou velocidade de forma justa.

In [16]:
# ==============================================================================
# CÉLULA 3: PRÉ-PROCESSAMENTO (SIMPLIFICADA)
# Responsabilidade: Transformar dados brutos em matrizes de treino/teste.
# ==============================================================================
print("\n--- FASE 2: PREPARAÇÃO DE DADOS ---")

# 1. Unir as bases e remover colunas inúteis
df = pd.concat([df1, df2, df3], ignore_index=True)
if 'Numero' in df.columns: df.drop(columns=['Numero'], inplace=True)

# 2. Seleção de colunas e Alvo
TARGET = 'Entorse'
cols = [TARGET, 'T0_T1_Match_Time_exposure', 'T0_T1_Training_Time_exposure', 
        'T0SRTMax', 'T0TTestMin', 'T0SJmMax', 'T0Veli', 'T0RazaoAADto', 'T0RazaoAAEsq']

df = df[[c for c in cols if c in df.columns]].copy()
df = df.fillna(df.median(numeric_only=True))

# 3. Definição de X/Y (0 = Saudável, 1 = Entorse)
y = df[TARGET].apply(lambda x: 1 if x > 1 else 0) 
X = pd.get_dummies(df.drop(columns=[TARGET]), drop_first=True)

# 4. Divisão e Escalonamento
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.75, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"✅ Dados Prontos para a Rede Neuronal!")


--- FASE 2: PREPARAÇÃO DE DADOS ---
✅ Dados Prontos para a Rede Neuronal!


### 4. Arquitetura da Rede Neuronal Profunda (Deep Learning)
Nesta fase, desenhamos a estrutura "cerebral" do **PlaySafe4All**. Optámos por uma arquitetura do tipo **Sequential** em funil, que processa a informação biomecânica de forma cada vez mais abstrata até chegar a uma decisão final.

* **Camada de Entrada (`Input`)**: Recebe as variáveis normalizadas (features) correspondentes aos testes de força, velocidade e exposição.
* **Camadas Densas (`Dense`) com Ativação ReLU**: Utilizamos camadas de 64, 32 e 16 neurónios. A função de ativação **ReLU** permite que a rede aprenda relações não-lineares complexas, como a interação entre o cansaço acumulado (exposure) e a falha de força explosiva.
* **Mecanismo de Regularização (`Dropout`)**: Inserimos camadas de Dropout (30% e 20%) que "desligam" neurónios aleatoriamente durante o treino. Isto força a rede a não depender de apenas um neurónio, prevenindo o *overfitting* e garantindo que o modelo generalize bem para novos atletas.
* **Camada de Saída (`Sigmoid`)**: A última camada utiliza a função Sigmoid para comprimir o resultado num valor entre 0 e 1, representando a probabilidade estatística de ocorrência de entorse.
* **Compilação Estratégica**: Utilizamos o otimizador **Adam** e a métrica de **Recall** como prioridade, assegurando que o treino da rede seja direcionado para a máxima segurança do desportista.

In [17]:
# ==============================================================================
# CÉLULA 4: ARQUITETURA DA REDE NEURONAL
# Responsabilidade: Definir a estrutura de camadas e neurónios (Deep Learning).
# ==============================================================================
print("--- FASE 3: CONSTRUÇÃO DA REDE NEURONAL ---")

# 1. Definir a arquitetura em funil
model_nn = Sequential([
    Input(shape=(X_train_scaled.shape[1],)), 
    
    # Camada 1: 64 neurónios com Dropout para evitar overfitting
    Dense(64, activation='relu'),
    Dropout(0.3), 
    
    # Camada 2: 32 neurónios
    Dense(32, activation='relu'),
    Dropout(0.2),
    
    # Camada 3: 16 neurónios
    Dense(16, activation='relu'),
    
    # Camada de Saída: Sigmoid para probabilidade de 0 a 1
    Dense(1, activation='sigmoid')
])

# 2. Compilação focada em Recall (Segurança)
model_nn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)

model_nn.summary()

--- FASE 3: CONSTRUÇÃO DA REDE NEURONAL ---


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                      │ (None, 64)                  │           3,648 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,273 (24.50 KB)

 Trainable params: 6,273 (24.50 KB)

 Non-trainable params: 0 (0.00 B)

### 5. Ciclo de Treino e Otimização de Pesos (Fitting)
Nesta fase, a Rede Neuronal inicia o processo de aprendizagem iterativa. O objetivo é ajustar os pesos de cada neurónio para que a previsão final seja o mais próxima possível da realidade biomecânica.

* **Equilíbrio de Classes (`class_weight`)**: Como as entorses são eventos menos frequentes no nosso dataset, calculamos pesos inversamente proporcionais à frequência de cada classe. Isto obriga a rede a "prestar mais atenção" aos casos de lesão, evitando que ela se torne preguiçosa e classifique todos os atletas como saudáveis.
* **Épocas de Treino (`epochs=50`)**: A rede passará por todo o conjunto de dados 50 vezes. Em cada passagem, ela utiliza o erro cometido para se auto-corrigir através do algoritmo de retropropagação (*Backpropagation*).
* **Processamento em Lote (`batch_size=8`)**: Os dados são processados em pequenos grupos de 8 amostras de cada vez. Isto permite um equilíbrio entre a estabilidade do treino e a velocidade de convergência.
* **Histórico de Aprendizagem**: O sistema armazena a evolução da precisão e do erro ao longo do tempo, permitindo-nos verificar se a rede está realmente a aprender padrões ou se está apenas a memorizar ruído.

In [18]:
# ==============================================================================
# CÉLULA 5: TREINO DA REDE NEURONAL
# Responsabilidade: Treinar o modelo com pesos equilibrados para as lesões.
# ==============================================================================
if 'model_nn' in locals():
    print("--- FASE 4: TREINO DA REDE NEURONAL ---")

    # Ajuste de pesos para dar importância às entorses (classe 1)
    counts = np.bincount(y_train)
    weight_for_0 = 1.0 / counts[0]
    weight_for_1 = 1.0 / counts[1]
    class_weights = {0: weight_for_0, 1: weight_for_1}

    # Treino
    history = model_nn.fit(
        X_train_scaled, y_train,
        epochs=50, 
        batch_size=8, 
        class_weight=class_weights,
        verbose=1 # Mostra o progresso do treino
    )
    print("\n✅ Treino da Rede Neuronal concluído!")
else:
    print("❌ Erro: A arquitetura (Célula 4) não foi definida.")

--- FASE 4: TREINO DA REDE NEURONAL ---
Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.5111 - loss: 0.0329 - recall: 0.3600
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4667 - loss: 0.0310 - recall: 0.4400 
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5333 - loss: 0.0296 - recall: 0.6000 
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5556 - loss: 0.0299 - recall: 0.5600 
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5778 - loss: 0.0294 - recall: 0.6400 
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6000 - loss: 0.0299 - recall: 0.6400
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7111 - loss: 0.0285 - recall: 0.7600 
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7333 - loss: 0.0281 - recall: 0.8800 
Epoch 9/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7556 - loss: 0.0267 - recall: 0.8000 
Epoch 10/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 

### 6. Avaliação Final e Validação de Segurança (Rede Neuronal)
Esta fase final submete a Rede Neuronal a um teste de rigor biomecânico, utilizando dados de teste que o modelo nunca visualizou, garantindo a integridade dos resultados e a sua capacidade de generalização.

* **Inferência de Probabilidades**: A saída da camada *Sigmoid* fornece um valor entre 0 e 1. Aplicamos um **Threshold de 0.5** (limiar de decisão): se a rede indicar mais de 50% de probabilidade, o sistema emite um alerta de risco de entorse.
* **Métricas de Performance Deep Learning**:
    * **Accuracy**: Eficácia global da rede na classificação de atletas.
    * **Recall (Sensibilidade)**: O indicador mestre do **PlaySafe4All**. Mede a capacidade da rede em identificar corretamente todos os atletas que sofreram efetivamente uma lesão.
    * **F1-Score**: O equilíbrio harmónico que garante que a rede não está apenas a "adivinhar" ao acaso.
* **Matriz de Confusão e Falsos Negativos (FN)**: 
    * Analisamos especificamente os **Falsos Negativos** (atletas em risco classificados como saudáveis). 
    * A validação do projeto depende de mantermos este erro no mínimo absoluto (FN ≤ 2), assegurando que a tecnologia é um suporte fiável para a equipa médica e de fisioterapia.
* **Latência de Resposta**: O tempo de execução em milissegundos valida a viabilidade do uso desta Rede Neuronal em dispositivos móveis ou sistemas de monitorização *on-field* em tempo real.

In [19]:
# ==============================================================================
# CÉLULA 6: AVALIAÇÃO FINAL (REDE NEURONAL)
# Responsabilidade: Validar o desempenho final e segurança da rede.
# ==============================================================================
if 'model_nn' in locals():
    print("\n--- FASE 5: AVALIAÇÃO FINAL (Rede Neuronal) ---")

    # ⏱️ Início da medição
    start_time = time.perf_counter()

    # 1. Obter probabilidades (Saída da Sigmoid: 0.0 a 1.0)
    y_probs = model_nn.predict(X_test_scaled)
    
    # 2. Transformar em 0 ou 1 (Threshold de 0.5)
    # Se a probabilidade for > 50%, prevemos Entorse (1)
    y_pred_nn = (y_probs > 0.5).astype(int).flatten()

    # 3. Cálculo das Métricas
    acc_nn = accuracy_score(y_test, y_pred_nn)
    rec_nn = recall_score(y_test, y_pred_nn)
    f1_nn = f1_score(y_test, y_pred_nn) # <--- Adicionado aqui
    cm_nn = confusion_matrix(y_test, y_pred_nn)
    
    # ⏱️ Fim da medição
    exec_time = (time.perf_counter() - start_time) * 1000

    # 4. Apresentação de Resultados
    print(f"📊 Precisão (Accuracy): {acc_nn:.4f}")
    print(f"🎯 Recall (Sensibilidade): {rec_nn:.4f}")
    print(f"🧪 F1-Score: {f1_nn:.4f}")
    print(f"⏱️ Tempo de Execução: {exec_time:.2f} ms")

    print("\nMatriz de Confusão (Rede Neuronal):")
    print(cm_nn)

    # 5. Erro Crítico (Falsos Negativos)
    fn_nn = cm_nn[1, 0]
    print(f"\n❌ Falsos Negativos (Casos perdidos): {fn_nn}")

    # Validação Final do Projeto
    if fn_nn <= 2:
        print("\n--- ✅ REDE NEURONAL VALIDADA PARA O PLAYSAFE4ALL ---")
    else:
        print(f"\n--- ⚠️ AVISO: FN={fn_nn}. O RECALL PRECISA DE MELHORIAS ---")
else:
    print("❌ Erro: O modelo não foi treinado. Por favor, execute a Célula 5 primeiro.")


--- FASE 5: AVALIAÇÃO FINAL (Rede Neuronal) ---
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
📊 Precisão (Accuracy): 0.5221
🎯 Recall (Sensibilidade): 0.4933
🧪 F1-Score: 0.5324
⏱️ Tempo de Execução: 200.02 ms

Matriz de Confusão (Rede Neuronal):
[[34 27]
 [38 37]]

❌ Falsos Negativos (Casos perdidos): 38

--- ⚠️ AVISO: FN=38. O RECALL PRECISA DE MELHORIAS ---
